# AdS4 Phase-Lock Identifiability Sweep (v9)

This notebook measures a phase-lock breakdown threshold by sweeping structured probe corruption.

We:
- remove direct metric supervision
- keep EE + WL + GEO + consistency structure
- vary corruption strength on the GEO probe
- measure reconstruction error and consistency drift

Goal:
- identify when stable reconstruction breaks down
- turn probe consistency into a measurable threshold


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# Ground-truth AdS4-style toy background
r = torch.linspace(1.05, 3.5, 240, device=device).view(-1, 1)

def f_true(r):
    return r**2 + 1 - 1/r

def g_true(r):
    return 1/(f_true(r) + 1e-6)

def ee_true(r):
    return torch.log(1.0 + 0.6 * f_true(r))

def wl_true(r):
    return torch.sqrt(f_true(r) + 1.8 * g_true(r) + 1e-6)

def geo_true(r):
    return torch.sqrt(f_true(r) + 0.7 * g_true(r) + 1e-6)

ee_obs = ee_true(r).detach()
wl_obs = wl_true(r).detach()
geo_obs = geo_true(r).detach()


In [ ]:
def structured_geo_target(r, corruption_strength=0.0):
    base = geo_true(r)
    if corruption_strength == 0.0:
        return base.detach()
    radial_bias = 1.0 + corruption_strength * (0.30 * (r - r.min()) / (r.max() - r.min()))
    oscillation = 1.0 + corruption_strength * 0.15 * torch.sin(2.2 * r)
    return (base * radial_bias * oscillation).detach()


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = MLP()
        self.g = MLP()

    def forward(self, r):
        f = F.softplus(self.f(r))
        g = F.softplus(self.g(r))
        return f, g


In [ ]:
def train(corruption_strength=0.0, epochs=1200, consistency_weight=0.15):
    m = Model().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    geo_target = structured_geo_target(r, corruption_strength)

    loss_hist = []
    consistency_hist = []

    for _ in range(epochs):
        opt.zero_grad()
        f, g = m(r)

        ee_pred = torch.log(1.0 + 0.6 * f)
        wl_pred = torch.sqrt(f + 1.8 * g + 1e-6)
        geo_pred = torch.sqrt(f + 0.7 * g + 1e-6)

        loss_ee = ((ee_pred - ee_obs)**2).mean()
        loss_wl = ((wl_pred - wl_obs)**2).mean()
        loss_geo = ((geo_pred - geo_target)**2).mean()
        loss_consistency = ((f * g - 1.0)**2).mean()

        loss = loss_ee + loss_wl + loss_geo + consistency_weight * loss_consistency
        loss.backward()
        opt.step()

        loss_hist.append(loss.item())
        consistency_hist.append(loss_consistency.item())

    with torch.no_grad():
        f_pred, g_pred = m(r)
        metric_err = ((f_pred - f_true(r)).abs() + (g_pred - g_true(r)).abs()) / 2.0
        abs_err_mean = metric_err.mean().item()
        abs_err_max = metric_err.max().item()
        consistency_final = consistency_hist[-1]

    return {
        'loss_hist': loss_hist,
        'consistency_hist': consistency_hist,
        'f_pred': f_pred,
        'g_pred': g_pred,
        'abs_err_mean': abs_err_mean,
        'abs_err_max': abs_err_max,
        'consistency_final': consistency_final,
    }


In [ ]:
# Sweep corruption strength
strengths = torch.linspace(0.0, 0.5, 11).tolist()
results = []

for s in strengths:
    out = train(corruption_strength=float(s), epochs=1200)
    results.append({
        'strength': float(s),
        'abs_err_mean': out['abs_err_mean'],
        'abs_err_max': out['abs_err_max'],
        'consistency_final': out['consistency_final'],
        'loss_hist': out['loss_hist'],
        'f_pred': out['f_pred'],
        'g_pred': out['g_pred'],
    })
    print(f"corruption={s:.2f} | mean err={out['abs_err_mean']:.6f} | max err={out['abs_err_max']:.6f} | consistency={out['consistency_final']:.6f}")


In [ ]:
# Identify a rough threshold where max error exceeds a chosen tolerance
tolerance = 0.10
threshold = None
for row in results:
    if row['abs_err_max'] > tolerance:
        threshold = row['strength']
        break

print('tolerance:', tolerance)
print('estimated breakdown threshold:', threshold)


In [ ]:
# Phase transition / identifiability figure
strength_vals = [row['strength'] for row in results]
mean_err_vals = [row['abs_err_mean'] for row in results]
max_err_vals = [row['abs_err_max'] for row in results]
consistency_vals = [row['consistency_final'] for row in results]

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(strength_vals, mean_err_vals, marker='o', label='mean abs error')
plt.axhline(tolerance, linestyle='--', label='tolerance')
if threshold is not None:
    plt.axvline(threshold, linestyle=':', label='threshold')
plt.xlabel('corruption strength')
plt.ylabel('mean abs error')
plt.title('identifiability sweep')
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(strength_vals, max_err_vals, marker='o', color='tab:red', label='max abs error')
plt.axhline(tolerance, linestyle='--', label='tolerance')
if threshold is not None:
    plt.axvline(threshold, linestyle=':', label='threshold')
plt.xlabel('corruption strength')
plt.ylabel('max abs error')
plt.title('breakdown threshold')
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(strength_vals, consistency_vals, marker='o', color='tab:green', label='consistency drift')
plt.xlabel('corruption strength')
plt.ylabel('final consistency loss')
plt.title('geometry class stability')
plt.legend()

plt.subplot(2, 2, 4)
plt.plot(results[0]['loss_hist'], label='clean')
mid_idx = len(results)//2
plt.plot(results[mid_idx]['loss_hist'], label=f"mid={results[mid_idx]['strength']:.2f}")
plt.plot(results[-1]['loss_hist'], label=f"high={results[-1]['strength']:.2f}")
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('training under corruption')
plt.legend()

plt.suptitle('AdS4 Phase-Lock Identifiability Sweep (v9)')
plt.tight_layout()
plt.savefig('ads4_phase_lock_v9_identifiability.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v9_identifiability.png')


In [ ]:
# Reconstruction comparison at low / threshold / high corruption
pick_low = 0
pick_mid = mid_idx
pick_high = len(results) - 1

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(r.cpu(), f_true(r).cpu(), label='f true')
plt.plot(r.cpu(), results[pick_low]['f_pred'].cpu(), '--', label=f"f @{results[pick_low]['strength']:.2f}")
plt.plot(r.cpu(), g_true(r).cpu(), label='g true')
plt.plot(r.cpu(), results[pick_low]['g_pred'].cpu(), ':', label=f"g @{results[pick_low]['strength']:.2f}")
plt.title('low corruption')
plt.legend(fontsize=8)

plt.subplot(1, 3, 2)
plt.plot(r.cpu(), f_true(r).cpu(), label='f true')
plt.plot(r.cpu(), results[pick_mid]['f_pred'].cpu(), '--', label=f"f @{results[pick_mid]['strength']:.2f}")
plt.plot(r.cpu(), g_true(r).cpu(), label='g true')
plt.plot(r.cpu(), results[pick_mid]['g_pred'].cpu(), ':', label=f"g @{results[pick_mid]['strength']:.2f}")
plt.title('mid corruption')
plt.legend(fontsize=8)

plt.subplot(1, 3, 3)
plt.plot(r.cpu(), f_true(r).cpu(), label='f true')
plt.plot(r.cpu(), results[pick_high]['f_pred'].cpu(), '--', label=f"f @{results[pick_high]['strength']:.2f}")
plt.plot(r.cpu(), g_true(r).cpu(), label='g true')
plt.plot(r.cpu(), results[pick_high]['g_pred'].cpu(), ':', label=f"g @{results[pick_high]['strength']:.2f}")
plt.title('high corruption')
plt.legend(fontsize=8)

plt.tight_layout()
plt.savefig('ads4_phase_lock_v9_reconstruction_examples.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v9_reconstruction_examples.png')
